In [1]:
# Cell 1: Start Spark session
import findspark
findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Arabic-AI-Detection-EDA") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"  Spark {spark.version} started")
print(f"  Spark UI: {spark.sparkContext.uiWebUrl}")

26/05/19 06:08:15 WARN Utils: Your hostname, localhost.localdomain resolves to a loopback address: 127.0.0.1; using 192.168.1.26 instead (on interface enp0s3)
26/05/19 06:08:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/19 06:08:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


  Spark 3.5.8 started
  Spark UI: http://192.168.1.26:4040


In [2]:
# Cell 2: Load datasets from HDFS
HDFS_RAW = "hdfs://localhost:9000/user/hadoop/arabic-ai-detection/raw"

df_polish = spark.read.option("header", "true").option("multiline", "true").option("escape", '"').csv(f"{HDFS_RAW}/by_polishing.csv")
df_title = spark.read.option("header", "true").option("multiline", "true").option("escape", '"').csv(f"{HDFS_RAW}/from_title.csv")
df_title_content = spark.read.option("header", "true").option("multiline", "true").option("escape", '"').csv(f"{HDFS_RAW}/from_title_and_content.csv")

# Row counts
counts = {
    "by_polishing": df_polish.count(),
    "from_title": df_title.count(),
    "from_title_and_content": df_title_content.count(),
}
total = sum(counts.values())
print("Dataset Row Counts")
print("-" * 40)
for split, n in counts.items():
    print(f"  {split:<25} {n:>6} rows")
print("-" * 40)
print(f"  {'TOTAL':<25} {total:>6} rows")

Dataset Row Counts
----------------------------------------
  by_polishing                2851 rows
  from_title                  2963 rows
  from_title_and_content      2574 rows
----------------------------------------
  TOTAL                       8388 rows


In [3]:
# Cell 3: Inspect schema
print("Schema of 'by_polishing':")
df_polish.printSchema()

Schema of 'by_polishing':
root
 |-- original_abstract: string (nullable = true)
 |-- allam_generated_abstract: string (nullable = true)
 |-- jais_generated_abstract: string (nullable = true)
 |-- llama_generated_abstract: string (nullable = true)
 |-- openai_generated_abstract: string (nullable = true)



In [4]:
# Cell 4: Show sample row
df_polish.show(1, truncate=200, vertical=True)

-RECORD 0-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 original_abstract         | كثيرا ما ارتبطت المصادر التاريخية في الأندلس خاصة منها كتب التراجم والفهرسات والبرامج وغيرها بدراسة حياة العلماء والرواة والقضاة والساسة ؛ وقد تطورت هذه المادة حتى ترك لنا المؤلفون الأندلسيون سلسلة... 
 allam_generated_abstract  | يتناول هذا البحث موضوع التعليم بين النساء الأندلسيات من خلال دراسة المصادر التاريخية المتعلقة بتراجم العلماء والرواة والقضاة والساسة. يركز البحث على إبراز دور المرأة العالمة ومساهمتها في الإنتاج ال... 
 jais_generated_abstract   | تدرس هذه الدراسة دور المرأة في التعليم في الأندلس من خلال مصادر تاريخية مثل كتب التراجم والفهارس والبرامج. على الرغم من أن هذه المصادر تركز بشكل أساسي على العلماء الذكور، إلا أنها توفر أيضا معلومات... 
 llama_generated_abstract  | يُقدم هذا البحث دراسة شاملة حول حالة التعليم عن

In [5]:
# Cell 5: Null counts per column
print("Null counts in 'by_polishing':")
df_polish.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_polish.columns
]).show(vertical=True, truncate=False)

Null counts in 'by_polishing':
-RECORD 0------------------------
 original_abstract         | 0   
 allam_generated_abstract  | 0   
 jais_generated_abstract   | 0   
 llama_generated_abstract  | 0   
 openai_generated_abstract | 0   



In [6]:
# Cell 6: Average abstract length per generator (in characters)
print("Average abstract length (characters):\n")
for col_name in df_polish.columns:
    avg_len = df_polish.select(F.avg(F.length(F.col(col_name)))).first()[0]
    if avg_len:
        print(f"  {col_name:<35}  →  {avg_len:>6.0f} chars")

Average abstract length (characters):

  original_abstract                    →     741 chars
  allam_generated_abstract             →     673 chars
  jais_generated_abstract              →     435 chars
  llama_generated_abstract             →     649 chars
  openai_generated_abstract            →    1068 chars


In [7]:
# Cell 7: Class balance check
n_total = total
n_human = n_total
n_ai = n_total * 4

print("Class Distribution (after reshape to long format)")
print("-" * 50)
print(f"  Human abstracts:  {n_human:>6} ({n_human/(n_human+n_ai)*100:.1f}%)")
print(f"  AI abstracts:     {n_ai:>6} ({n_ai/(n_human+n_ai)*100:.1f}%)")
print(f"  TOTAL:            {n_human+n_ai:>6}")
print()

Class Distribution (after reshape to long format)
--------------------------------------------------
  Human abstracts:    8388 (20.0%)
  AI abstracts:      33552 (80.0%)
  TOTAL:             41940



In [8]:
# Cell 8: Combine splits and tag with generation method
df_polish_t = df_polish.withColumn("generation_method", F.lit("by_polishing"))
df_title_t = df_title.withColumn("generation_method", F.lit("from_title"))
df_tc_t = df_title_content.withColumn("generation_method", F.lit("from_title_and_content"))

df_all = df_polish_t.unionByName(df_title_t).unionByName(df_tc_t)
print(f"Combined dataset: {df_all.count()} rows")

print("\nRows per generation method:")
df_all.groupBy("generation_method").count().orderBy("generation_method").show()

Combined dataset: 8388 rows

Rows per generation method:
+--------------------+-----+
|   generation_method|count|
+--------------------+-----+
|        by_polishing| 2851|
|          from_title| 2963|
|from_title_and_co...| 2574|
+--------------------+-----+



In [9]:
# Cell 9: Word count comparison
print("Average WORD count per source\n")
print("-" * 60)

for col_name in ["original_abstract", "allam_generated_abstract",
                 "jais_generated_abstract", "llama_generated_abstract",
                 "openai_generated_abstract"]:
    stats = df_all.select(
        F.avg(F.size(F.split(F.col(col_name), r"\s+"))).alias("avg"),
        F.min(F.size(F.split(F.col(col_name), r"\s+"))).alias("min"),
        F.max(F.size(F.split(F.col(col_name), r"\s+"))).alias("max")
    ).first()
    label = col_name.replace("_generated_abstract", "").replace("_abstract", "")
    print(f"  {label:<10}  avg: {stats['avg']:>6.1f}   min: {stats['min']:>4}   max: {stats['max']:>4}")

Average WORD count per source

------------------------------------------------------------


26/05/19 06:08:29 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

  original    avg:  119.2   min:   75   max:  294
  allam       avg:   91.9   min:   30   max: 2796
  jais        avg:   77.7   min:   30   max:  435
  llama       avg:  101.7   min:   33   max:  276
  openai      avg:  134.6   min:   85   max:  224


In [10]:
# Cell 10: Save the combined raw dataset as Parquet
HDFS_RAW_PARQUET = "hdfs://localhost:9000/user/hadoop/arabic-ai-detection/raw_combined.parquet"

df_all.write.mode("overwrite").parquet(HDFS_RAW_PARQUET)
print(f"  Saved combined raw dataset as Parquet to:\n  {HDFS_RAW_PARQUET}")

# Verify
df_check = spark.read.parquet(HDFS_RAW_PARQUET)
print(f"\nVerification: {df_check.count()} rows loaded back from Parquet")
df_check.printSchema()

  Saved combined raw dataset as Parquet to:
  hdfs://localhost:9000/user/hadoop/arabic-ai-detection/raw_combined.parquet

Verification: 8388 rows loaded back from Parquet
root
 |-- original_abstract: string (nullable = true)
 |-- allam_generated_abstract: string (nullable = true)
 |-- jais_generated_abstract: string (nullable = true)
 |-- llama_generated_abstract: string (nullable = true)
 |-- openai_generated_abstract: string (nullable = true)
 |-- generation_method: string (nullable = true)



Notebook ready. Run spark.stop() in the last cell when done.
